## Title
Exploracion de datos
### By:
Dovaribi Carupia Yagari

### Date:
2026-08-20

### Description:
Crear una nueva rama de git (Usar Gitflow) y Crear un notebook para la exploración inicial de los datos

    Tomar como ejemplo los pasos de: https://joserzapata.github.io/post/ciencia-datos-proyecto-python/2-exploration/

el objetivo es realizar una exploración general de datos para verificar los tipos de datos con el fin de comprender de las características y el esquema de datos y si es posible solucionar problemas básicos relacionados. realizar

    Descripción general de los datos
    Unificar la forma como se representan los valores Nulos
    Convertir los datos en su tipo correcto (numéricos, categóricos, booleanos, fechas, etc) y corrección de los datos si es necesario, para eu cada columna tenga un tipo de dato uniforme.
    Almacenar el dataset final en un formato adecuado como .parquet



## 📚 Import  libraries

In [1]:
import logging
from pathlib import Path

import pandas as pd

## 💾 Load data

In [24]:
# Configuración de logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Rutas
DATA_DIR_RAW = Path("../..") / "data" / "01_raw"
DATA_DIR_INTER = Path("../..") / "data" / "02_intermediate"

# Asegurar que la carpeta de destino exista
DATA_DIR_INTER.mkdir(parents=True, exist_ok=True)

FILE_PATH_RAW = DATA_DIR_RAW / "Pacientes_porblemas_higado_india.csv"
FILE_PATH_PARQUET = DATA_DIR_INTER / "pacientes_higado_exploracion.parquet"

# Definimos estrictamente las columnas para limpiar las 1024 columnas fantasma
VALID_COLUMNS = [
    "Age",
    "Gender",
    "Total_Bilirubin",
    "Direct_Bilirubin",
    "Alkaline_Phosphotase",
    "Alamine_Aminotransferase",
    "Aspartate_Aminotransferase",
    "Total_Protiens",
    "Albumin",
    "Albumin_and_Globulin_Ratio",
    "Dataset",
]

## Data description

In [33]:
# 1. Carga de datos filtrando las columnas correctas
logging.info("Cargando datos crudos...")
df = pd.read_csv(FILE_PATH_RAW, usecols=VALID_COLUMNS)

# Renombrar la columna objetivo para mayor claridad de negocio
logging.info("Renombrando la variable objetivo...")

df = df.rename(columns={"Dataset": "Diagnosis"})


print("\n" + "=" * 50)
print("INFORMACIÓN GENERAL DEL DATASET")
print("=" * 50)
df.info()

print("\n" + "=" * 50)
print("MUESTRA")
print("=" * 50)
display(df.sample(5))

2026-08-20 20:55:26,949 - INFO - Cargando datos crudos...
2026-08-20 20:55:26,963 - INFO - Renombrando la variable objetivo...



INFORMACIÓN GENERAL DEL DATASET
<class 'pandas.DataFrame'>
RangeIndex: 663 entries, 0 to 662
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Age                         662 non-null    float64
 1   Gender                      655 non-null    str    
 2   Total_Bilirubin             657 non-null    float64
 3   Direct_Bilirubin            656 non-null    float64
 4   Alkaline_Phosphotase        646 non-null    float64
 5   Alamine_Aminotransferase    644 non-null    float64
 6   Aspartate_Aminotransferase  651 non-null    float64
 7   Total_Protiens              656 non-null    float64
 8   Albumin                     661 non-null    float64
 9   Albumin_and_Globulin_Ratio  659 non-null    float64
 10  Diagnosis                   648 non-null    float64
dtypes: float64(10), str(1)
memory usage: 60.0 KB

MUESTRA


,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Diagnosis
448,48.0,Female,0.8,0.2,142.0,26.0,25.0,6.0,2.6,0.7,1.0
215,66.0,Male,0.6,0.2,100.0,17.0,148.0,5.0,3.3,1.9,2.0
204,21.0,Male,0.7,0.2,135.0,27.0,26.0,6.4,3.3,1.0,2.0
75,29.0,Female,0.7,0.1,162.0,52.0,41.0,5.2,2.5,0.9,2.0
261,33.0,Male,1.5,7.0,505.0,205.0,140.0,7.5,3.9,1.0,1.0


## NULL VALUES

In [27]:
# 2. Unificación de la representación de valores nulos
logging.info("Iniciando análisis y unificación de valores nulos...")

# Imprimimos los valores únicos de columnas categóricas (como Gender) para cazar nulos ocultos
print("Valores únicos en 'Gender' antes de limpiar:")
print(df["Gender"].unique())

# Lista de posibles nulos ocultos detectados en sistemas transaccionales
valores_nulos_ocultos = ["", " ", "NA", "N/A", "NULL", "null", "?", "-"]

# Reemplazamos esos valores por el nulo oficial de Pandas
df = df.replace(valores_nulos_ocultos, pd.NA)

print("\n" + "=" * 40)
print("CONTEO REAL DE VALORES NULOS POR COLUMNA")
print("=" * 40)
# Mostramos solo las columnas que efectivamente tienen datos faltantes
display(df.isna().sum()[df.isna().sum() > 0])

2026-08-20 20:27:11,109 - INFO - Iniciando análisis y unificación de valores nulos...


Valores únicos en 'Gender' antes de limpiar:
<ArrowStringArray>
['Female', 'Male', nan]
Length: 3, dtype: str

CONTEO REAL DE VALORES NULOS POR COLUMNA


Age                            1
Gender                         8
Total_Bilirubin                6
Direct_Bilirubin               7
Alkaline_Phosphotase          17
Alamine_Aminotransferase      19
Aspartate_Aminotransferase    12
Total_Protiens                 7
Albumin                        2
Albumin_and_Globulin_Ratio     4
Diagnosis                     15
dtype: int64

## Convert data types

In [29]:
# 3. Conversión de tipos de datos correctos
logging.info("Ajustando tipos de datos...")

# 'Int64' (con mayúscula) permite números enteros mezclados con valores nulos (pd.NA)
tipos_correctos = {
    "Gender": "category",
    "Diagnosis": "category",
    "Age": "Int64",
}

df = df.astype(tipos_correctos)

print("\nTipos de datos actualizados:")
display(df.dtypes)

2026-08-20 20:53:41,083 - INFO - Ajustando tipos de datos...



Tipos de datos actualizados:


Age                              Int64
Gender                        category
Total_Bilirubin                float64
Direct_Bilirubin               float64
Alkaline_Phosphotase           float64
Alamine_Aminotransferase       float64
Aspartate_Aminotransferase     float64
Total_Protiens                 float64
Albumin                        float64
Albumin_and_Globulin_Ratio     float64
Diagnosis                     category
dtype: object

## Save dataframe with datatypes

In [34]:
# 4. Almacenar el dataset final en formato Parquet
logging.info("Guardando dataset en formato Parquet...")

try:
    df.to_parquet(FILE_PATH_PARQUET, index=False, engine="pyarrow")
    logging.info("Archivo guardado exitosamente en: %s", FILE_PATH_PARQUET)
except Exception as e:
    logging.exception("Error al intentar guardar el archivo Parquet.")
    raise ValueError(f"Error en la exportación: {e}") from e

# Validación de la exportación leyendo el archivo nuevo
df_parquet = pd.read_parquet(FILE_PATH_PARQUET)
print("\nDimensiones del nuevo archivo Parquet:", df_parquet.shape)

2026-08-20 20:57:51,541 - INFO - Guardando dataset en formato Parquet...
2026-08-20 20:57:51,576 - INFO - Archivo guardado exitosamente en: ..\..\data\02_intermediate\pacientes_higado_exploracion.parquet



Dimensiones del nuevo archivo Parquet: (663, 11)


## 📝 Resumen del Análisis de Variables y Tipos de Datos

Tras la inspección y el ajuste inicial, la estructura de nuestro dataset queda documentada de la siguiente manera para la fase de modelado:

**1. Variables Categóricas (Cualitativas):**
* **`Gender`:** Sexo del paciente. Se mantiene como categoría de texto en esta etapa para facilitar la lectura de las gráficas. En la fase de *Feature Engineering* será codificada numéricamente (ej. 0 para Femenino, 1 para Masculino).
* **`Diagnosis` (Target):** Variable objetivo del negocio. Es categórica ya que representa clases fijas establecidas por los médicos:
  * **1**: Paciente diagnosticado con enfermedad hepática.
  * **2**: Paciente sano (sin enfermedad hepática).

**2. Variables Numéricas Discretas (Enteros):**
* **`Age`:** Edad del paciente. Se forzó al tipo de dato `Int64` de Pandas. Esto permite mantener su naturaleza lógica de número entero (sin decimales) y, al mismo tiempo, soportar la presencia de valores nulos reales (`pd.NA`) sin corromper la columna.

**3. Variables Numéricas Continuas (Floats):**
* **El resto de las variables** (`Total_Bilirubin`, `Alkaline_Phosphotase`, `Total_Protiens`, etc.) se mantienen como `float64`. 
* *Justificación:* Estas columnas representan resultados de exámenes de química sanguínea y mediciones de laboratorio. Son variables inherentemente continuas que requieren alta precisión decimal (por ejemplo, una proporción de albúmina de 0.9 mg/dL). Forzarlas a números enteros redondearía los valores, destruyendo la precisión médica de los datos y arruinando la capacidad predictiva del modelo.